In [1]:
import os
import os.path
import pickle
import pandas as pd
import numpy as np
from tqdm import tqdm
import re

In [2]:
import os
import sys
from pathlib import Path

# === CONFIGURATION ===
# Choose which dataset to run on: "val" or "test"
DATASET_MODE = "test"  # Change to "test" for final submission

# Set to True to rebuild indices from CSV (required on first run)
# Set to False to load cached indices (faster for subsequent runs)
FORCE_REBUILD_INDICES = False

# Detect environment
KAGGLE_ENV = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if KAGGLE_ENV:
    # Kaggle paths
    DATA_PATH = Path("/kaggle/input/omnilex-data")
    MODEL_PATH = Path("/kaggle/input/llama-model")
    OUTPUT_PATH = Path("/kaggle/working")
    INDEX_PATH = Path("/kaggle/input/omnilex-indices")
    sys.path.insert(0, "/kaggle/input/omnilex-utils")
else:
    # Local development paths
    REPO_ROOT = Path(".").resolve().parent
    DATA_PATH = REPO_ROOT / "data"
    MODEL_PATH = REPO_ROOT / "models"
    OUTPUT_PATH = REPO_ROOT / "output"
    INDEX_PATH = REPO_ROOT / "data" / "processed"

# CSV corpus files for index building
LAWS_CSV = DATA_PATH / "laws_de.csv"
COURTS_CSV = DATA_PATH / "court_considerations.csv"

# Index cache paths
LAWS_INDEX_PATH = INDEX_PATH / "laws_index.pkl"
COURTS_INDEX_PATH = INDEX_PATH / "courts_index.pkl"

# Derived paths based on DATASET_MODE
QUERY_FILE = DATA_PATH / f"{DATASET_MODE}.csv"
IS_VALIDATION_MODE = DATASET_MODE == "val"

# Create output directory
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
INDEX_PATH.mkdir(parents=True, exist_ok=True)

print(f"Environment: {'Kaggle' if KAGGLE_ENV else 'Local'}")
print(f"Dataset mode: {DATASET_MODE}")
print(f"Query file: {QUERY_FILE}")
print(f"Validation mode: {IS_VALIDATION_MODE}")
print(f"Force rebuild indices: {FORCE_REBUILD_INDICES}")
print(f"\nCorpus files:")
print(f"  Laws CSV: {LAWS_CSV} ({LAWS_CSV.stat().st_size / 1e6:.1f} MB)" if LAWS_CSV.exists() else f"  Laws CSV: {LAWS_CSV} (NOT FOUND)")
print(f"  Courts CSV: {COURTS_CSV} ({COURTS_CSV.stat().st_size / 1e9:.2f} GB)" if COURTS_CSV.exists() else f"  Courts CSV: {COURTS_CSV} (NOT FOUND)")
print(f"\nIndex cache: {INDEX_PATH}")

Environment: Local
Dataset mode: test
Query file: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/test.csv
Validation mode: False
Force rebuild indices: False

Corpus files:
  Laws CSV: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/laws_de.csv (73.0 MB)
  Courts CSV: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/court_considerations.csv (2.28 GB)

Index cache: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/processed


In [3]:
from FlagEmbedding import FlagReranker, BGEM3FlagModel

dense_model = BGEM3FlagModel('/root/.cache/modelscope/hub/models/BAAI/bge-m3', use_fp16=True)
reranker = FlagReranker('/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3', use_fp16=True, normalize=True) # Setting use_fp16 to True speeds up computation with a slight performance degradation


libgomp: Invalid value for environment variable OMP_NUM_THREADS

libgomp: Invalid value for environment variable OMP_NUM_THREADS


In [4]:
court_consideration_df = pd.read_csv("../data/court_considerations.csv")
court_consideration_d = {}
for citation, text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist()):
    # if citation in court_consideration_d:
    #     court_consideration_d[citation] = court_consideration_d[citation] + '\n\n' + text
    # else:
    #     court_consideration_d[citation] = text
    court_consideration_d[citation] = text

law_df = pd.read_csv("../data/laws_de.csv")
law_d = dict(zip(law_df['citation'].tolist(), law_df['text'].tolist()))

test_df = pd.read_csv('../data/test_rewrite_001.csv')

_d = {}
for _, row in test_df.iterrows():
    if row['query_id'] not in _d:
        _d[row['query_id']] = [row['query']]
    else:
        _d[row['query_id']].append(row['query'])
test_dict = {k: v for k, v in sorted(_d.items())}
    

court_doc = [{'citation':citation, 'text':text} for citation,text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist())]
law_doc = [{'citation':citation, 'text':text} for citation,text in zip(law_df['citation'].tolist(), law_df['text'].tolist())]

print("data loaded")

data loaded


In [5]:
import dense_index
from dense_index import DenseIndex

print(dense_model.normalize_embeddings)
court_dense_index = DenseIndex(dense_model, "../data/processed/_dense_sparse_court", court_doc)
court_dense_index.info()

law_dense_index = DenseIndex(dense_model, "../data/processed/_dense_law", law_doc)
law_dense_index.info()

True
DenseIndex.embeddings:  (2107648, 1024)
[dense_index] documents.len: 1985178 parent_idx.len: 2107648
DenseIndex.embeddings:  (176032, 1024)
[dense_index] documents.len: 175933 parent_idx.len: 176032


In [6]:
from sparse_index import SparseIndex

court_sparse_index = SparseIndex(dense_model, "../data/processed/_dense_sparse_court", court_doc)
court_sparse_index.load()

In [7]:
import citation_utils
import rerank_utils
import rrf

RECALL_COUNT=1000
RERANK_COUNT=100
NN = 10

id_l = []
citation_l = []

def recall_rerank_get_nn(query, court_sparse_search_l):
    
    court_rerank_l = rerank_utils.rerank_by_dense_batch_chunked(reranker, query, court_sparse_search_l, RERANK_COUNT, 20, 384, 128)
    court_rerank_citation_l = [c['citation'] for c,_ in court_rerank_l]

    court_nn_doc_l = []
    court_nn_doc_l.extend([doc for doc,_ in court_rerank_l])
    
    ret_l = court_dense_index.search_batch(court_rerank_citation_l, NN)
    for ret in ret_l:
        court_nn_doc_l.extend(ret)
    court_nn_rerank_l = rerank_utils.rerank_by_dense_batch_chunked(reranker, query, court_nn_doc_l, len(court_nn_doc_l), 20, 384, 128)
    return [c['citation'] for c, _ in court_nn_rerank_l]

for query_id, query_l in tqdm(test_dict.items(), total=len(test_dict)):
    ranked_l_l = []
    for query in query_l:
        court_sparse_search_l = court_sparse_index.search(query, RECALL_COUNT)
        ret = recall_rerank_get_nn(query, court_sparse_search_l)
        ranked_l_l.append(ret)

        court_dense_search_l = court_dense_index.search(query, RECALL_COUNT)
        ret = recall_rerank_get_nn(query, court_dense_search_l)
        ranked_l_l.append(ret)

    print(f"{query_id} court dense-sparse search done.")

    query_result = rrf.compute2(ranked_l_l, k=60, top_k=100)

    raw_hits = citation_utils.BFS_citation(court_consideration_d, law_d, query_result, max_level=2)
    
    law_hits = [hits for hits in raw_hits if hits['citation'] in law_d]

    print("raw_hits.len:", len(raw_hits), ", law_hits.len:", len(law_hits))

    query_result_top20 = query_result[:20]

    law_rerank_l = rerank_utils.rerank_by_dense_batch_chunked(reranker, query, law_hits, 20, 20, 384, 128)

    # 去重
    citations = [r for r in query_result_top20]
    for _law, score in law_rerank_l:
        citations.append(_law['citation'])
    citations = list(set(citations))
    id_l.append(query_id)
    citation_l.append(';'.join(citations))
    print(query_id, len(citations))

result_df = pd.DataFrame({'query_id':id_l, 'predicted_citations':citation_l})
result_df.to_csv("../data/result.csv", index=False)

  0%|          | 0/40 [00:00<?, ?it/s]You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


test_001 court dense-sparse search done.
raw_hits.len: 133 , law_hits.len: 25


  2%|▎         | 1/40 [03:43<2:25:03, 223.18s/it]

test_001 40
test_002 court dense-sparse search done.
raw_hits.len: 158 , law_hits.len: 42


  5%|▌         | 2/40 [07:43<2:27:48, 233.37s/it]

...

test_038 40
test_039 court dense-sparse search done.
raw_hits.len: 168 , law_hits.len: 47


 98%|█████████▊| 39/40 [2:29:03<03:45, 225.55s/it]

test_039 40
test_040 court dense-sparse search done.
raw_hits.len: 172 , law_hits.len: 39


100%|██████████| 40/40 [2:32:52<00:00, 229.31s/it]

test_040 40
